# Notebook limpio de manin, aquí probaremos nuevas estrategias con orderflow

In [1]:
import numpy as np
import pandas as pd
import sys
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sys.path.append(os.path.abspath("../src"))
from data_cleaner import DataCleanerConfig, DataCleaner, preprocess_data


## Importamos los datos 

In [2]:
cfg_5m = DataCleanerConfig(
    source="alpaca",
    symbol=["QQQ", "TLT", "VXX", "BNDX"],
    interval="5m",
    start_date="2022-01-01",
    end_date="2026-02-02",
    
)

cleaner = DataCleaner(cfg_5m)
df_raw = cleaner.cargar_datos()

df = preprocess_data(df_raw, "qqq")

df.head()


c:\Users\jorge\OneDrive\Escritorio\TRADING_ALG\src\data_cleaner.py:242: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df["close"].pct_change()


,datetime,open_bndx,open_qqq,open_tlt,open_vxx,high_bndx,high_qqq,high_tlt,high_vxx,low_bndx,...,log_return_lag5,vol_rolling,rsi,macd,macd_signal,ema_9,ema_21,ema_50,ema_100,ema_200
0,2022-01-03 14:25:00+00:00,55.06,399.38,146.55,18.2100,55.060,399.41,146.600,18.30,55.06,...,-0.000275,0.000512,21.777778,-0.000707,-0.000518,399.110000,399.110000,399.110000,399.110000,399.110000
1,2022-01-03 14:30:00+00:00,55.03,399.05,146.41,18.2700,55.060,400.68,146.500,18.29,55.01,...,0.000601,0.000921,50.991501,-0.000474,-0.000509,399.372000,399.229091,399.161373,399.135941,399.123035
2,2022-01-03 14:35:00+00:00,55.02,400.44,146.36,18.1200,55.035,401.53,146.595,18.16,55.01,...,-0.000025,0.000983,60.246913,-0.000158,-0.000439,399.709600,399.395537,399.235829,399.174041,399.142308
3,2022-01-03 14:40:00+00:00,55.03,401.07,146.57,18.0701,55.040,401.26,146.800,18.15,55.03,...,0.000326,0.001024,57.547170,-0.000008,-0.000353,399.879680,399.501397,399.287757,399.201486,399.156414
4,2022-01-03 14:45:00+00:00,55.03,400.57,146.53,18.1100,55.050,400.75,146.940,18.56,55.03,...,-0.000050,0.001669,36.489076,-0.000359,-0.000354,399.547244,399.384679,399.245786,399.182001,399.147072


# Definimos el target
*df["ret_fwd"] = df["close"].shift(-H) / df["close"] - 1* -> Crea una nueva columna ret_fwd y calcula el retorno porcentual (ejemplo: si hoy close es 100, en H velas close = 101 -> 1%)

*df.loc[df["ret_fwd"] >  thr, "y"] = 1* -> Crea la etiqueta binaria Y.  y = 1 si el retorno futuro es positivo y 0 si es negativo

# Resumen: 
Calculamos el retorno futuro a H velas (ret_fwd).

Solo etiquetamos como sube (1) si el movimiento supera +thr y como baja (0) si baja más de -thr.

Todo lo que sea “ruido” (entre -thr y +thr) lo descartamos para entrenar con ejemplos más claros.

In [3]:
H = 24          # horizonte 
thr = 0.0005    # 0.05% = 5 bps 

df = df.copy()
df["ret_fwd"] = df["close"].shift(-H) / df["close"] - 1

df["y"] = np.nan
df.loc[df["ret_fwd"] >  thr, "y"] = 1
df.loc[df["ret_fwd"] < -thr, "y"] = 0

df = df.dropna(subset=["ret_fwd", "y"]).reset_index(drop=True)
df["y"] = df["y"].astype(int)


*df["ret_1"] = df["close"] / df["close"].shift(1) - 1* -> Crea el retorno de 1 vela hacia atrás (ret_1)

Dividimos train y test en 70% train, 30% test

*majority = int(train["y"].mean() >= 0.5)* -> Decide cuál es la clase más común en el train

*test["pred_majority"] = majority* -> Predice siempre esa clase (0 o 1) para todas las filas del test

*test["pred_mom1"] = (test["ret_1"] > 0).astype(int)* -> Si la vela anterior subió (ret_1 > 0) predice 1, si no predice 0.

In [16]:
# Feature ultra simple (solo pasado)
df = df.copy()
df["ret_1"] = df["close"] / df["close"].shift(1) - 1
df = df.dropna(subset=["ret_1"]).reset_index(drop=True)

# Split temporal (sin shuffle)
split = int(len(df) * 0.70)
train = df.iloc[:split].copy()
test  = df.iloc[split:].copy()

# Baseline 1: predice siempre la clase mayoritaria del train
majority = int(train["y"].mean() >= 0.5)
test["pred_majority"] = majority

# Baseline 2: momentum 1 vela (si la última vela subió, predice 1)
test["pred_mom1"] = (test["ret_1"] > 0).astype(int)


In [75]:
# ===== Features SUPER simples (solo pasado) =====
df = df.copy()

# MAs: distancia % al MA (20 y 50)
df["ma20_dist"] = df["close"] / df["close"].rolling(20).mean().shift(1) - 1
df["ma50_dist"] = df["close"] / df["close"].rolling(50).mean().shift(1) - 1

# RSI (7 y 14) versión simple con medias rolling
d = df["close"].diff()
up = d.clip(lower=0)
down = (-d).clip(lower=0)

rs7  = up.rolling(7).mean()  / down.rolling(7).mean()
rs14 = up.rolling(14).mean() / down.rolling(14).mean()

df["rsi7"]  = (100 - 100/(1+rs7)).shift(1)
df["rsi14"] = (100 - 100/(1+rs14)).shift(1)

# MACD hist en % (12,26,9)
ema12 = df["close"].ewm(span=12, adjust=False).mean()
ema26 = df["close"].ewm(span=26, adjust=False).mean()
macd = ema12 - ema26
sig  = macd.ewm(span=9, adjust=False).mean()
df["macd_hist_pct"] = ((macd - sig) / df["close"]).shift(1)

# Pasamos features a train/test (sin re-split)
feat_cols = ["ret_1", "ma20_dist", "ma50_dist", "rsi7", "rsi14", "macd_hist_pct"]
for c in feat_cols:
    train[c] = df.loc[train.index, c].values
    test[c]  = df.loc[test.index,  c].values

train_ml = train.dropna(subset=feat_cols + ["y"]).copy()
test_ml  = test.dropna(subset=feat_cols + ["y"]).copy()


In [ ]:
from sklearn.metrics import balanced_accuracy_score

Xtr, ytr = train_ml[feat_cols], train_ml["y"].astype(int)
Xte, yte = test_ml[feat_cols],  test_ml["y"].astype(int)

sc = StandardScaler()
lr = LogisticRegression(max_iter=200)

lr.fit(sc.fit_transform(Xtr), ytr)
pred_logit = lr.predict(sc.transform(Xte))

print("ACC logit:", round((pred_logit == yte).mean(), 4))
print("BAL_ACC logit:", round(balanced_accuracy_score(yte, pred_logit), 4))

print("ACC majority:", round((test.loc[test_ml.index, "pred_majority"].astype(int) == yte).mean(), 4))
print("ACC mom1:",     round((test.loc[test_ml.index, "pred_mom1"].astype(int) == yte).mean(), 4))


(51854, 75) (22245, 85)


,datetime,y,ret_1,ma20_dist,ma50_dist,rsi7,rsi14,macd_hist_pct
51904,2024-10-31 16:00:00+00:00,0,-0.000062,-0.001925,-0.013734,53.488372,38.340192,0.000543
51905,2024-10-31 16:05:00+00:00,0,0.000371,-0.001338,-0.012847,43.831169,39.533239,0.000615
51906,2024-10-31 16:10:00+00:00,0,0.000433,-0.000707,-0.011912,41.156463,40.707351,0.000691


In [78]:
from sklearn.metrics import balanced_accuracy_score

# 1) entreno logit
sc = StandardScaler()
lr = LogisticRegression(max_iter=200)
lr.fit(sc.fit_transform(train_ml[feat_cols]), train_ml["y"].astype(int))

pred_logit = lr.predict(sc.transform(test_ml[feat_cols])).astype(int)

# 2) comparo vs baselines en el MISMO subset (test_ml)
y = test_ml["y"].astype(int).values
p_maj = test.loc[test_ml.index, "pred_majority"].astype(int).values
p_mom = test.loc[test_ml.index, "pred_mom1"].astype(int).values

res = pd.DataFrame({
    "acc": [
        (p_maj == y).mean(),
        (p_mom == y).mean(),
        (pred_logit == y).mean()
    ],
    "bal_acc": [
        balanced_accuracy_score(y, p_maj),
        balanced_accuracy_score(y, p_mom),
        balanced_accuracy_score(y, pred_logit)
    ]
}, index=["majority", "mom1", "logit"]).sort_values("bal_acc", ascending=False)

# (opcional) guardo la predicción
test.loc[test_ml.index, "pred_logit"] = pred_logit

res


,acc,bal_acc
mom1,0.512969,0.511847
logit,0.554192,0.511055
majority,0.558058,0.500000


In [79]:
from sklearn.metrics import balanced_accuracy_score
import numpy as np

# mini valid temporal dentro de train (último 20%)
s = int(len(train_ml) * 0.8)
tr = train_ml.iloc[:s]
va = train_ml.iloc[s:]

sc = StandardScaler()
lr = LogisticRegression(max_iter=200, class_weight="balanced")
lr.fit(sc.fit_transform(tr[feat_cols]), tr["y"].astype(int))

# busco threshold que maximiza bal_acc en valid
p_va = lr.predict_proba(sc.transform(va[feat_cols]))[:,1]
ths = np.linspace(0.3, 0.7, 81)
ba  = [balanced_accuracy_score(va["y"].astype(int), (p_va > t).astype(int)) for t in ths]
tbest = ths[int(np.argmax(ba))]

# eval en test
p_te = lr.predict_proba(sc.transform(test_ml[feat_cols]))[:,1]
pred = (p_te > tbest).astype(int)
y = test_ml["y"].astype(int).values

p_mom = test.loc[test_ml.index, "pred_mom1"].astype(int).values

print("tbest:", round(float(tbest),3), "bal_acc(val):", round(float(max(ba)),4))
print("BAL_ACC mom1:",  round(balanced_accuracy_score(y, p_mom), 4))
print("BAL_ACC logit:", round(balanced_accuracy_score(y, pred), 4))


tbest: 0.52 bal_acc(val): 0.5174
BAL_ACC mom1: 0.5118
BAL_ACC logit: 0.5035
